# Pre-label farm frames with the MVBroTrack detector

Runs on a free Colab GPU (Runtime > Change runtime type > T4 GPU).

What it does:
1. collects the frames from the `train`, `val` and `test` folders on Google Drive
   (the old split is ignored: it had frames of the same clip in train and test),
2. reads the recording date and time from the timestamp burned into each frame,
3. lets the MVBroTrack `yolo11x_1920.pt` model draw a box around every bird,
4. saves labels, preview images and a CVAT import archive back to Drive.

Afterwards you only correct the boxes in CVAT; see `docs/finetune.md`.

## Settings

`DRIVE_DIR` must point to the folder that contains `train`, `val`, `test` and `MVBroTrack_v2`.
If the folder was shared with you, first add it to your Drive
(right click > Organize > Add shortcut) so it appears under `MyDrive`.

`PLACEMENT_DATE` is day 0 of the flock (the day the chicks arrived). It is used for the bird
age and the growth phase of every frame. Fix it if the value below is wrong; the import cell can be
re-run at any time.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/poultry_2026"
PLACEMENT_DATE = "2025-11-13"   # PLACEHOLDER: check with the farm records
CAMERA = "cam1"
WORK = "/content/labeling"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## Install

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!git clone -q https://github.com/hastitajik/broiler-health-early-warning /content/repo
!pip -q install -e /content/repo ultralytics pytesseract

## Find the model

In [ ]:
import glob
hits = glob.glob(f"{DRIVE_DIR}/**/trained_models/yolo11x_1920.pt", recursive=True)
assert hits, "yolo11x_1920.pt not found under DRIVE_DIR"
MODEL = hits[0]
print(MODEL)

## 1. Import the frames (reads the date from the image)

In [ ]:
!rm -rf {WORK}
!cd /content/repo && pews import-frames "{DRIVE_DIR}/train" "{DRIVE_DIR}/val" "{DRIVE_DIR}/test" \
    --placement-date {PLACEMENT_DATE} --camera {CAMERA} --out {WORK}

## 2. Draft the boxes

In [ ]:
!cd /content/repo && pews prelabel --dir {WORK} --weights "{MODEL}" --imgsz 1920 --conf 0.25 --device 0

## 3. Look at the drafts

Green boxes = birds found by the model. Look for birds without a box (often chicks in week one
and birds packed along the walls) and boxes on things that are not birds.

In [ ]:
import csv, random
from IPython.display import Image, display
rows = list(csv.DictReader(open(f"{WORK}/manifest.csv")))
by_phase = {}
for r in rows:
    by_phase.setdefault(r["phase"], []).append(r)
for phase, rs in sorted(by_phase.items()):
    for r in random.sample(rs, min(2, len(rs))):
        print(phase, "| age", r["age_days"], "days |", r["ts"])
        display(Image(f"{WORK}/previews/{r['image'].rsplit('.', 1)[0]}.jpg", width=900))

In [ ]:
# boxes per frame: a quick sanity check of the drafts, by growth phase
import statistics
for phase, rs in sorted(by_phase.items()):
    n = [sum(1 for _ in open(f"{WORK}/labels/{r['image'].rsplit('.', 1)[0]}.txt")) for r in rs]
    print(f"{phase:9s} frames={len(n):3d}  boxes per frame: median={statistics.median(n):.0f} min={min(n)} max={max(n)}")

## 4. Save to Drive

In [ ]:
OUT = f"{DRIVE_DIR}/labeling"
!mkdir -p "{OUT}"
!cp -r {WORK}/images {WORK}/labels {WORK}/previews {WORK}/manifest.csv {WORK}/cvat_prelabels_yolo1.1.zip "{OUT}/"
!ls -la "{OUT}"

## Next: correct the boxes in CVAT

1. app.cvat.ai (free) > Create task > label `broiler` (and `dead`) > upload all files from `labeling/images`.
2. Task > Actions > Upload annotations > format **YOLO 1.1** > `cvat_prelabels_yolo1.1.zip`.
3. Fix the boxes (see `docs/labeling_guide.md`).
4. Export dataset > **YOLO 1.1** and put the `.txt` files in `labeling/labels_corrected/` on Drive.
5. Then: `pews build-dataset`, `pews train`, `pews evaluate` (see `docs/finetune.md`).